In [5]:
# 1. Install the required tools FIRST
!pip install -q python-docx "google-genai<2.0.0" "pydantic==2.12.3"

# 2. Unzip the uploaded file quietly and overwrite existing files (-o)
!unzip -o -q "[PUB] India_runs_data_and_ai_challenge.zip"

# 1. Install the required tools FIRST
!pip install -q python-docx google-genai "pydantic==2.12.3"

import json
from docx import Document
from google import genai
from google.genai import types

# 3. Read the text files from the extracted folder
def get_text_from_word(filename):
    doc = Document(filename)
    return '\n'.join([para.text for para in doc.paragraphs])

folder = '[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/'

job_desc_text = get_text_from_word(folder + 'job_description.docx')
signals_text = get_text_from_word(folder + 'redrob_signals_doc.docx')

print("Files successfully loaded into memory!")

# 4. Setup the GenAI Client
GOOGLE_API_KEY = "AQ.Ab8RN6JoJ9aIbQAnfzW2HvyeZY889uShug3h1P6_UwCwIhwuMA"
client = genai.Client(api_key=GOOGLE_API_KEY)

prompt = f"""
You are an expert technical recruiter. Read the following Job Description and the 'Red Rob Signals' (rules for finding a great candidate).

Extract the core needs of this role into a clean JSON format with these exact keys:
- "mandatory_skills": (a list of the top 5 most important skills)
- "experience_level": (the career level needed, like Junior, Mid, or Senior)
- "core_mission": (a 2-sentence summary of what this person needs to achieve)

Job Description:
{job_desc_text}

Red Rob Signals:
{signals_text}
"""

print("Asking AI to analyze the job requirements... please wait...")

# 5. Generate the structured JSON response
response = client.models.generate_content(
    model='gemini-2.5-flash', # <--- Updated model name
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
    ),
)

print("\n--- AI Job Analysis ---")
print(response.text)

Files successfully loaded into memory!
Asking AI to analyze the job requirements... please wait...

--- AI Job Analysis ---
{
  "mandatory_skills": [
    "Production experience with embeddings-based retrieval systems (handling drift, index refresh, regressions).",
    "Production experience with vector databases or hybrid search infrastructure (operational experience).",
    "Strong Python for writing production-quality code.",
    "Expertise in designing and interpreting evaluation frameworks for ranking systems (e.g., NDCG, MRR, A/B testing).",
    "Ability to architect, build, and rapidly ship end-to-end ML-driven ranking and matching systems, integrating modern ML techniques like LLMs and fine-tuning."
  ],
  "experience_level": "Senior",
  "core_mission": "The core mission is to own and enhance Redrob's intelligence layer by designing, building, and deploying cutting-edge ML-driven ranking, retrieval, and matching systems. This involves rapidly shipping production-ready solutions,

In [6]:

print("Asking AI to analyze the job requirements... please wait...")

# 4. Generate the structured JSON response
response = client.models.generate_content(
    model='gemini-3.5-flash',
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
    ),
)

print("\n--- AI Job Analysis ---")
print(response.text)

Asking AI to analyze the job requirements... please wait...

--- AI Job Analysis ---
{
  "mandatory_skills": [
    "Embeddings-based retrieval systems",
    "Vector databases and hybrid search infrastructure",
    "Python",
    "Evaluation frameworks for ranking systems (NDCG, MRR, MAP)",
    "Production machine learning systems deployment"
  ],
  "experience_level": "Senior",
  "core_mission": "Own and architect the intelligence layer of Redrob's talent platform, specifically driving the ranking, retrieval, and matching systems that connect recruiters with candidates at scale. Quickly design, ship, and iteratively optimize production-ready ML systems while building out the foundational evaluation infrastructure and mentoring a growing engineering team."
}


In [7]:
from sentence_transformers import SentenceTransformer, util
import json

print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# 1. Target vector
target_text = "Senior engineer. " + " ".join([
    "Embeddings-based retrieval systems",
    "Vector databases and hybrid search",
    "Ranking and recommendation systems",
    "Evaluation frameworks (NDCG, MRR, A/B testing)",
    "Python"
]) + " Own and scale the intelligence layer candidate-job matching, ranking, and retrieval systems."

target_embedding = embedder.encode(target_text, convert_to_tensor=True)

# 2. Load the candidates
folder = '[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/'
candidates_list = []
with open(folder + 'candidates.jsonl', 'r') as file:
    for line in file:
        candidates_list.append(json.loads(line))

def flatten_text(data):
    if isinstance(data, str):
        return data
    elif isinstance(data, list):
        return " ".join(flatten_text(item) for item in data)
    elif isinstance(data, dict):
        return " ".join(flatten_text(value) for value in data.values())
    elif data is None:
        return ""
    else:
        return str(data)

# 3. Extract all text first
print("Extracting text from all candidates...")
experience_texts = []
for candidate in candidates_list:
    text = flatten_text(candidate.get('profile_summary', '')) + " " + \
           flatten_text(candidate.get('experience', '')) + " " + \
           flatten_text(candidate.get('skills', ''))
    experience_texts.append(text)

# 4. BATCH ENCODE (This is the massive speed boost!)
print(f"Encoding {len(experience_texts)} candidates in batches...")
candidate_embeddings = embedder.encode(experience_texts, batch_size=32, show_progress_bar=True, convert_to_tensor=True)

# 5. Calculate all scores at once
print("Calculating similarity scores...")
# This does the math for all candidates simultaneously against the target
cosine_scores = util.cos_sim(target_embedding, candidate_embeddings)[0]

# 6. Match scores back to candidate names
candidate_scores = []
for i, candidate in enumerate(candidates_list):
    score = cosine_scores[i].item()

    candidate_id = flatten_text(candidate.get('candidate_id', 'Unknown'))

    name = "Unknown"
    if 'personal_info' in candidate and isinstance(candidate['personal_info'], dict):
        name = candidate['personal_info'].get('name', 'Unknown')
    else:
        name = candidate.get('name', 'Unknown')

    candidate_scores.append({
        "candidate_id": candidate_id,
        "name": flatten_text(name),
        "score": score
    })

# 7. Sort and Print
top_candidates = sorted(candidate_scores, key=lambda x: x['score'], reverse=True)

print("\n🚀 --- TOP 5 CANDIDATES --- 🚀")
for i in range(5):
    c = top_candidates[i]
    print(f"{i+1}. {c['name']} (ID: {c['candidate_id']}) - Score: {c['score']:.4f}")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Extracting text from all candidates...
Encoding 100000 candidates in batches...


Batches:   0%|          | 0/3125 [00:00<?, ?it/s]

Calculating similarity scores...

🚀 --- TOP 5 CANDIDATES --- 🚀
1. Unknown (ID: CAND_0075439) - Score: 0.5806
2. Unknown (ID: CAND_0093193) - Score: 0.5722
3. Unknown (ID: CAND_0081846) - Score: 0.5705
4. Unknown (ID: CAND_0082618) - Score: 0.5586
5. Unknown (ID: CAND_0074225) - Score: 0.5561


In [8]:
import json

# 1. Prepare the final list according to standard schema
final_submission = []

for rank, candidate in enumerate(top_candidates):
    # We format this exactly how the judges usually want it
    submission_entry = {
        "rank": rank + 1,
        "candidate_id": candidate["candidate_id"],
        "name": candidate["name"],
        "match_score": round(candidate["score"], 4) # Rounding to 4 decimal places for clean data
    }
    final_submission.append(submission_entry)

# 2. Save it to a JSON file
output_filename = "final_submission.json"

with open(output_filename, 'w') as outfile:
    json.dump(final_submission, outfile, indent=4)

print(f"🎉 Success! Your top {len(final_submission)} candidates have been saved to '{output_filename}'.")

# 3. Print the first entry so you can verify it matches the schema rules
print("\nHere is what your final output looks like:")
print(json.dumps(final_submission[0], indent=2))

🎉 Success! Your top 100000 candidates have been saved to 'final_submission.json'.

Here is what your final output looks like:
{
  "rank": 1,
  "candidate_id": "CAND_0075439",
  "name": "Unknown",
  "match_score": 0.5806
}
